In [2]:
import pandas as pd
import joblib
import json

# Auto-select best model from metadata
with open("qb_model_metadata.json", "r") as f:
    metadata_qb = json.load(f)

model_name_qb = metadata_qb["model_name"]
print(f"Using {model_name_qb.upper()} model (RF RMSE: {metadata_qb['rf_rmse']:.4f}, XGB RMSE: {metadata_qb['xgb_rmse']:.4f})")

model_qb = joblib.load(f"best_{model_name_qb}_qb_model.joblib")
with open(f"{model_name_qb}_qb_feature_cols.json", "r") as f:
    feature_cols_qb = json.load(f)

# Load prediction data
pred_qb = pd.read_csv("../data/processed/qb_pred_combined.csv")

# Generate predictions
identifiers_qb = pred_qb[["player_id", "player_display_name", "college_flag"]]
X_pred_qb = pred_qb[feature_cols_qb]

predictions_qb = model_qb.predict(X_pred_qb)

results_qb = identifiers_qb.copy()
results_qb["predicted_weekly_ppr"] = predictions_qb.round(2)
results_qb = results_qb.sort_values("predicted_weekly_ppr", ascending=False).reset_index(drop=True)

print(results_qb.head(20))

results_qb.to_csv("../data/processed/qb_predictions_2026.csv", index=False)
print("\nExport complete")
print(f"Total players predicted: {len(results_qb)}")
print(f"NFL players: {(results_qb['college_flag'] == 0).sum()}")
print(f"CFB players: {(results_qb['college_flag'] == 1).sum()}")

Using XGB model (RF RMSE: 5.1060, XGB RMSE: 5.0931)
     player_id player_display_name  college_flag  predicted_weekly_ppr
0   00-0034857          Josh Allen             0             18.469999
1   00-0039732              Bo Nix             0             18.360001
2   00-0033873     Patrick Mahomes             0             17.820000
3   00-0039851          Drake Maye             0             17.760000
4   00-0036355      Justin Herbert             0             17.650000
5   00-0036971     Trevor Lawrence             0             17.080000
6   00-0039918      Caleb Williams             0             17.010000
7   00-0033077        Dak Prescott             0             16.900000
8   00-0026498    Matthew Stafford             0             16.889999
9   00-0033106          Jared Goff             0             16.799999
10  00-0036442          Joe Burrow             0             16.040001
11  00-0035228        Kyler Murray             0             15.690000
12  00-0034855      Baker